In [39]:
import CircuitBuilder
import numpy as np
import pickle as pkl
import os
from tqdm import tqdm

In [37]:
# Default 832 code

# 54627381 
def c832(er = 0, nrounds = 1):
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    # State preparation
    cb.al('R 0 1 2 3 4 5 6 7') # Physical qubits
    cb.al('R 8 9 10 11 12') # Ancilla qubits
    cb.set_ancilla_qubits(qubitslistz = [8, 9, 10, 11], qubitslistx = [12])
    cb.al('H 12')
    # Inject error
    #cb.al('X 2')
    # Warmup round
    cb.time_tick()
    cb.couple_z_check([0, 4, 5, 6], True)
    cb.time_tick()
    cb.couple_z_check([1, 5, 0, 4], True)
    cb.time_tick()
    cb.couple_z_check([2, 6, 1, 0], True)
    cb.time_tick()
    cb.couple_z_check([3, 7, 4, 2], True)
    cb.time_tick()
    cb.couple_x_check([0], True)
    cb.time_tick()
    cb.couple_x_check([1], True)
    cb.time_tick()
    cb.couple_x_check([2], True)
    cb.time_tick()
    cb.couple_x_check([3], True)
    cb.time_tick()
    cb.couple_x_check([4], True)
    cb.time_tick()
    cb.couple_x_check([5], True)
    cb.time_tick()
    cb.couple_x_check([6], True)
    cb.time_tick()
    cb.couple_x_check([7], True)
    cb.al('MR(0) 8 9 10 11')
    cb.al('H 12')
    cb.al('MR(0) 12')
    cb.al('H 12')
    # Noise rounds
    cb.begin_loop(nrounds)
    cb.time_tick()
    cb.couple_z_check([0, 4, 5, 6], True)
    cb.time_tick()
    cb.couple_z_check([1, 5, 0, 4], True)
    cb.time_tick()
    cb.couple_z_check([2, 6, 1, 0], True)
    cb.time_tick()
    cb.couple_z_check([3, 7, 4, 2], True)
    cb.time_tick()
    cb.couple_x_check([0], True)
    cb.time_tick()
    cb.couple_x_check([1], True)
    cb.time_tick()
    cb.couple_x_check([2], True)
    cb.time_tick()
    cb.couple_x_check([3], True)
    cb.time_tick()
    cb.couple_x_check([4], True)
    cb.time_tick()
    cb.couple_x_check([5], True)
    cb.time_tick()
    cb.couple_x_check([6], True)
    cb.time_tick()
    cb.couple_x_check([7], True)
    cb.al('MR(0) 8 9 10 11')
    cb.add_detectors([[-1, -6], [-2, -7], [-3, -8], [-4, -9]])
    cb.al('H 12')
    cb.al('MR(0) 12')
    cb.al('H 12')
    cb.add_detectors([[-1, -6]])
    cb.end_loop()
    # Final measurement of all physical qubits
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')

    return cb
# 1234
#     1234
# 23  41
# 3 4 2 1
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])

def windowFiller(schedule, nwindow):
    thiswindow = []
    for row in schedule:
        if nwindow in row:
            thiswindow.append(np.where(row == nwindow)[0][0])
        else:
            thiswindow.append(None)
    return thiswindow

def c832scheduleable(er = 0, nrounds = 1, zschedule = defaultzschedule, xschedule = defaultxschedule):
    zschedule, xschedule = np.asarray(zschedule), np.asarray(xschedule)
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    cb.al('R 0 1 2 3 4 5 6 7')
    #cb.al('X 0 1') # Flip some physical qubits to start, still must be in codespace
    nancillas = len(zschedule) + len(xschedule)
    ancillastring = 'R ' + ' '.join([f'{i}' for i in range(8, 8 + nancillas)])
    cb.al(ancillastring)
    cb.set_ancilla_qubits(qubitslistz = [8 + i for i in range(len(zschedule))], qubitslistx = [8 + len(zschedule) + i for i in range(len(xschedule))])
    #print(cb.ancilla_qubitsz, cb.ancilla_qubitsx)
    cb.al(f'H {cb.ancilla_qubitsx[0]}')
    # Warmup round
    for windowz in range(1, np.max(zschedule) + 1):
        cb.time_tick()
        #print(windowFiller(zschedule, windowz))
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    # Noise rounds
    cb.begin_loop(nrounds)
    for windowz in range(1, np.max(zschedule) + 1):
        cb.time_tick()
        cb.couple_z_check(windowFiller(zschedule, windowz), True)
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
    cb.end_loop()
    # Final measurement of all physical qubits
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')
    return cb
    

def detector_builder(mseq, syndromelength = 5, nphysicalqubits = 8, nrounds = 1, mode = 'binary'):
    if len(mseq) != syndromelength * (nrounds + 1) + nphysicalqubits:
        raise ValueError(f"Measurement sequence length must be equal to syndromelength * nrounds + nphysicalqubits")
    syndromes, finalmeasurement = mseq[:-nphysicalqubits], mseq[-nphysicalqubits:]
    if mode == 'string':
        det = []
        for i in range(nrounds):
            det.append(''.join([str(0 if x == y else 1) for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])]))
    elif mode == 'binary':
        det = np.zeros(nrounds * syndromelength, dtype = np.bool)
        for i in range(nrounds):
            det[i*syndromelength:(i+1)*syndromelength] = np.array([0 if x == y else 1 for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])])
    else:
        raise ValueError(f"Invalid mode: {mode}")

    return det, finalmeasurement
    
def passes_postselection(detector):
    return np.sum(detector) < 1

def toDecimal(binary):
    return int(binary, 2)

In [35]:
nrounds = [1, 2, 4, 6, 8]
ers = [0.0001, 0.000316, 0.001, 0.00316, 0.01]
shots = 1000000
def run_simulation(zschedule = defaultzschedule, xschedule = defaultxschedule, name = 'default'):   
    # Keep track of:
    nshots = np.zeros((len(nrounds), len(ers)), dtype = np.int32)
    detectorfired = np.zeros((len(nrounds), len(ers)), dtype = np.int32)
    readout = np.zeros((len(nrounds), len(ers), 2**8), dtype = np.int32)
    readoutpostselect = np.zeros((len(nrounds), len(ers), 2**8), dtype = np.int32)

    for i in tqdm(range(len(nrounds))):
        for j in tqdm(range(len(ers))):
            cb = c832scheduleable(ers[j], nrounds[i], zschedule=zschedule, xschedule=xschedule)
            sample = cb.sample(nshots = shots)
            for row in sample:
                det, finalmeasurement = detector_builder(row[0], len(defaultzschedule) + 1, 8, nrounds[i], mode = 'binary')
                readoutDecimal = toDecimal(finalmeasurement)
                nshots[i, j] += row[1]
                readout[i, j, readoutDecimal] += row[1]
                if passes_postselection(det):
                    readoutpostselect[i, j, readoutDecimal] += row[1]
                else:
                    detectorfired[i, j] += row[1]
    output = np.zeros((len(nrounds), len(ers), 514), dtype = np.int32)
    output[:, :, 0:256] = readout
    output[:, :, 256:512] = readoutpostselect
    output[:, :, 512] = nshots
    output[:, :, 513] = detectorfired
    output.tofile(f'Data/output_{name}.bin')    


In [36]:
cb = c832scheduleable(0.0001, 1, zschedule=defaultzschedule, xschedule=defaultxschedule)
sample = cb.sample(100000)
for row in sample:
    print(row, detector_builder(row[0], len(defaultzschedule) + 1, 8, 1, mode = 'string')[0])

('000000000000000000', 24239) ['00000']
('000000000000000001', 5) ['00000']
('000000000000000010', 15) ['00000']
('000000000000000100', 9) ['00000']
('000000000000000111', 2) ['00000']
('000000000000001000', 5) ['00000']
('000000000000001111', 1) ['00000']
('000000000000010000', 13) ['00000']
('000000000000011111', 1) ['00000']
('000000000000100000', 13) ['00000']
('000000000000111111', 2) ['00000']
('000000000001000000', 8) ['00000']
('000000000001111111', 15) ['00000']
('000000000010000000', 15) ['00000']
('000000000010111111', 12) ['00000']
('000000000011011111', 16) ['00000']
('000000000011100000', 2) ['00000']
('000000000011101111', 16) ['00000']
('000000000011110000', 1) ['00000']
('000000000011110111', 15) ['00000']
('000000000011111011', 12) ['00000']
('000000000011111100', 1) ['00000']
('000000000011111101', 8) ['00000']
('000000000011111110', 8) ['00000']
('000000000011111111', 24448) ['00000']
('000000000100000000', 108) ['00001']
('000000000100000001', 12) ['00001']
('00000

In [38]:
run_simulation(name = 'defaultnoiselessX')

100%|██████████| 5/5 [01:41<00:00, 20.26s/it]


In [5]:
schedulenames = [f for f in os.listdir('schedules') if os.path.isfile(os.path.join('schedules', f))]
print(schedulenames[0][9:17])

10100000
